# Extract small core subvolumes

The original slicing implementation is shared with `preprocessing/core_subvolume_extraction.py`. Outputs use the model loader's XYZ storage convention. Set `storage_order="ZYX"` for sources using the original notebook's ZYX convention.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "preprocessing").is_dir() else Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from preprocessing.core_subvolume_extraction import extract_subvolumes

file_path = PROJECT_ROOT / "data" / "core256.raw"
output_dir = PROJECT_ROOT / "outputs" / "subvolumes_64"
shape = (256, 256, 256)  # Source dimensions in storage_order
block_size = (64, 64, 64)
step_size = 64
storage_order = "XYZ"  # Use ZYX for the original notebook's source convention

blocks = extract_subvolumes(
    file_path, output_dir, shape, block_size, (step_size,) * 3,
    dtype="uint16", byte_order="little", storage_order=storage_order,
)
print(f"Saved {len(blocks)} subvolumes in {output_dir}; output storage order is XYZ")


# Extract a large core region

Crop coordinates follow the source storage order. The shared function checks the source size/bounds and writes XYZ output without loading the whole source into RAM.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "preprocessing").is_dir() else Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from preprocessing.core_subvolume_extraction import extract_crop

file_path = PROJECT_ROOT / "data" / "source_large.raw"
output_file = PROJECT_ROOT / "outputs" / "extracted_regions" / "cropped_xyz.raw"
shape = (1900, 2800, 2800)  # Source Z, Y, X dimensions
start = (128, 512, 512)  # Source Z, Y, X crop origin
extract_size = (1024, 1024, 1536)  # Source Z, Y, X crop shape

saved_shape = extract_crop(
    file_path, output_file, shape, start, extract_size,
    dtype="uint16", byte_order="little", storage_order="ZYX",
)
print(f"Saved {output_file}; XYZ output shape={saved_shape}")


# MS-SWD


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
from skimage.transform import resize
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. 3D MS-SWD algorithm
# ==========================================

def get_patches_3d(volume, patch_size=7, max_patches=1000):
    """
    Randomly extract 3D voxel patches from a volume.
    """
    d, h, w = volume.shape
    np.random.seed(42) # Use repeatable random patch locations
    
    # Randomly generate patch start coordinates
    zs = np.random.randint(0, d - patch_size + 1, max_patches)
    ys = np.random.randint(0, h - patch_size + 1, max_patches)
    xs = np.random.randint(0, w - patch_size + 1, max_patches)
    
    patches = []
    for z, y, x in zip(zs, ys, xs):
        # Extract 7x7x7 voxel patches
        patch = volume[z:z+patch_size, y:y+patch_size, x:x+patch_size]
        # Flatten each patch to a vector of length 7*7*7 = 343
        patches.append(patch.flatten())
        
    return np.array(patches)

def compute_swd_3d(patches_A, patches_B, num_projections=128):
    """Calculate the sliced Wasserstein distance between two 3D patch-feature distributions."""
    if patches_A.shape[0] == 0 or patches_B.shape[0] == 0:
        return 0.0
    
    dim = patches_A.shape[1] # The feature dimension is patch_size cubed
    projections = np.random.randn(dim, num_projections)
    projections /= np.linalg.norm(projections, axis=0, keepdims=True)
    
    proj_A = np.dot(patches_A, projections) 
    proj_B = np.dot(patches_B, projections)
    
    proj_A.sort(axis=0)
    proj_B.sort(axis=0)
    
    w_dist = np.mean(np.abs(proj_A - proj_B))
    return w_dist

def compute_ms_swd_3d(vol1, vol2, scales=[1, 0.5], patch_size=7, max_patches=500):
    """
    Calculate multiscale SWD between two 3D core volumes.
    The default uses scales 1 and 0.5 to limit the cost of 3D resizing.
    """
    ms_distance = 0.0
    for scale in scales:
        if scale != 1.0:
            d, h, w = vol1.shape
            # Resize the 3D volume
            scaled_vol1 = resize(vol1, (int(d * scale), int(h * scale), int(w * scale)), anti_aliasing=True)
            scaled_vol2 = resize(vol2, (int(d * scale), int(h * scale), int(w * scale)), anti_aliasing=True)
        else:
            scaled_vol1, scaled_vol2 = vol1, vol2
            
        patches_1 = get_patches_3d(scaled_vol1, patch_size, max_patches)
        patches_2 = get_patches_3d(scaled_vol2, patch_size, max_patches)
        
        ms_distance += compute_swd_3d(patches_1, patches_2)
        
    return ms_distance / len(scales)

# ==========================================
# 2. Load 3D RAW volumes
# ==========================================

def load_raw_3d_volumes(folder_path, shape=(256, 256, 256)):
    """
    Load all .raw files in a directory as uint16 and normalize their intensities.
    """
    volumes = []
    if not os.path.exists(folder_path):
        print(f"Warning: directory not found: {folder_path}")
        return volumes

    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.raw')])
    print(f"Loading 3D RAW volumes from {folder_path}; found {len(files)} files...")
    
    for f in files:
        img_path = os.path.join(folder_path, f)
        # 1. Read the uint16 binary data
        vol1D = np.fromfile(img_path, dtype=np.uint16)
        
        # Check that the file size matches the expected 256-cubed volume
        expected_size = shape[0] * shape[1] * shape[2]
        if vol1D.size != expected_size:
            print(f"  --> Skipping {f}: voxel count ({vol1D.size}) does not match expected shape {shape}.")
            continue
            
        # 2. Reshape to a 3D array
        vol3D = vol1D.reshape(shape)
        
        # 3. Normalize using the uint16 maximum of 65535
        # Convert to float32 and normalize to [0,1] for distance calculations
        vol3D_norm = vol3D.astype(np.float32) / 65535.0 
        
        volumes.append(vol3D_norm)
        
    return volumes

# ==========================================
# 3. Distance calculation and MDS visualization
# ==========================================

def calculate_distance_matrix(image_dict):
    """Calculate the distance matrix."""
    labels = []
    flat_volumes = []
    
    for class_name, vols in image_dict.items():
        for vol in vols:
            flat_volumes.append(vol)
            labels.append(class_name)
            
    n_samples = len(flat_volumes)
    dist_matrix = np.zeros((n_samples, n_samples))
    
    total_calcs = n_samples * (n_samples - 1) // 2
    print(f"\nComputing a {n_samples}x{n_samples} distance matrix with {total_calcs} pairwise calculations...")
    print("Extracting 3D patches may take some time...")
    
    current_calc = 0
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            # Calculate 3D MS-SWD
            dist = compute_ms_swd_3d(flat_volumes[i], flat_volumes[j])
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
            
            current_calc += 1
            print(f"  Progress: {current_calc}/{total_calcs} complete")
            
    return dist_matrix, labels

def plot_mds_density_contour(dist_matrix, labels):
    """Plot MDS density contours with settings for small sample sizes."""
    print("\nComputing the MDS embedding and plotting density contours...")
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
    coords_2d = mds.fit_transform(dist_matrix)
    
    import pandas as pd
    df = pd.DataFrame({
        'MDS Dimension 1': coords_2d[:, 0],
        'MDS Dimension 2': coords_2d[:, 1],
        'Method': labels
    })
    
    plt.figure(figsize=(9, 7))
    palette = sns.color_palette("Set1", len(set(labels)))
    
    # Increase bw_adjust for small samples (8 volumes) to smooth the density contours
    sns.kdeplot(
        data=df, x='MDS Dimension 1', y='MDS Dimension 2', 
        hue='Method', fill=True, alpha=0.15, linewidths=2, 
        palette=palette, levels=4, bw_adjust=1.5
    )
    
    # Emphasize sample points because density estimates are less reliable for small samples
    sns.scatterplot(
        data=df, x='MDS Dimension 1', y='MDS Dimension 2',
        hue='Method', palette=palette, s=150, alpha=0.9, 
        edgecolor='black', legend=True
    )
    
    plt.title('3D MS-SWD-MDS Density Map of Digital Cores', fontsize=15, fontweight='bold')
    plt.xlabel('MDS Dimension 1', fontsize=12)
    plt.ylabel('MDS Dimension 2', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. Run the analysis
# ==========================================
if __name__ == "__main__":
    gen_path = r"../../data/generated_binary_256"
    test_path = r"../../data/real_binary_256"
    
    # Set your volume shape
    VOLUME_SHAPE = (256, 256, 256)
    
    # Load 3D volumes
    test_volumes = load_raw_3d_volumes(test_path, shape=VOLUME_SHAPE)
    gen_volumes = load_raw_3d_volumes(gen_path, shape=VOLUME_SHAPE)
    
    if not test_volumes or not gen_volumes:
        print("Error: no data loaded; check the input paths and formats.")
    else:
        # Build the dataset mapping
        image_dataset = {
            'TI (Test Set)': test_volumes,
            'Reconstructed (Gen)': gen_volumes
        }
        
        # Run the evaluation
        distance_matrix, class_labels = calculate_distance_matrix(image_dataset)
        plot_mds_density_contour(distance_matrix, class_labels)